# 第 1 章 调用大模型 API

## 小明的烦恼

小明是一名大学生，正在自学编程。和很多同学一样，他遇到不懂的问题，第一反应就是打开 ChatGPT 问一问。

用了一段时间，小明发现聊天机器人确实很好用，但也有不少让人头疼的地方：

- **它记不住我**。每次开一个新对话，都要重新告诉它“我学到第几章了”“我更喜欢简短的解释”。
- **它不知道今天几号**。小明想让它帮忙排一个期末复习计划，可它连今天是星期几都说不准。
- **它没读过我的教材**。问它“我们课本里的某个概念是什么意思”，它只能按自己的理解回答，跟课本对不上。
- **它只会说，不会做**。小明想让它“三天后提醒我复习”，它只能回一句“好的，记得复习哦”，然后什么也不会发生。

这些问题，光靠换一个更聪明的模型是解决不了的。我们需要在大模型外面，再搭一套程序：帮它**记住**事情、给它**工具**去查日期和资料、让它学会**规划**和**检查**自己的工作。这样的程序，就叫**智能体（Agent）**。

> 智能体 = 大模型（大脑）+ 记忆 + 工具 + 规划
>
> 大模型负责“想”，智能体负责“想好以后去做”。

## 这本书要做什么

这本书就是要从零开始，一步步帮小明搭一个**学习助手智能体**。每一章都会给它加一项新本领：

| 章节 | 学习助手学会了什么 |
|---|---|
| 第 1–4 章 | 会聊天：调用大模型 API，写出第一个对话循环 |
| 第 5–8 章 | 有记性：记住对话，保存下来，下次接着聊 |
| 第 9–12 章 | 会动手：调用工具，连接远程的 MCP 工具服务器 |
| 第 13–16 章 | 有知识：从教材里查资料（RAG），长期记住小明的情况 |
| 第 17–20 章 | 会思考：推理、做计划、检查并修改自己的答案 |
| 第 21–24 章 | 能协作：工作流、多个智能体分工合作、安全和可靠性 |

所有代码都是普通的 Python，不依赖任何智能体框架。我们的目标不是做一个功能最强的产品，而是**看清楚智能体每一步是怎么工作的**。学完以后，再去看 LangGraph、OpenAI Agents SDK 这些框架，你会发现它们做的事情，你都亲手写过一遍。

第一步，先让我们的程序能跟大模型说上话。

## 从聊天网页到 API

我们平时用的 ChatGPT 网页，是**给人用的**：你在输入框里打字，它把回答显示在屏幕上。

但我们要写的是程序，程序没法去点网页。所以 OpenAI 还提供了另一个入口：**API（应用程序接口）**。程序通过网络把问题发给 OpenAI 的服务器，服务器把模型的回答发回来。背后用的是同一类模型，只是“说话的方式”从网页换成了代码。

```
网页：  人  ──打字──>  ChatGPT 网页  ──>  模型
API：  程序 ──请求──>  OpenAI API    ──>  模型
```

要使用 API，需要先做三件准备工作：注册账号并充值、创建 API Key、把 Key 保存到电脑的环境变量里。

## 准备工作 1：注册 OpenAI 开发者账号

1. 打开 OpenAI 开发者平台 [platform.openai.com](https://platform.openai.com)，注册或登录。可以直接用 ChatGPT 的账号登录。
2. 在设置里找到 **Billing（账单）**，绑定付款方式并充值一小笔钱。

有两点要注意：

- **API 和 ChatGPT 会员是分开收费的。** 就算你订阅了 ChatGPT Plus，API 也要单独充值。
- **API 按用量收费，单位是 token。** token 可以粗略理解为“一小段文字”，一个汉字大约是一到两个 token。我们这本书用的 `gpt-5.4-mini` 很便宜：问一个简单的问题，花费不到一美分。把书里的例子全部跑一遍，也花不了多少钱。

## 准备工作 2：创建 API Key

API Key 是一串以 `sk-` 开头的字符，相当于你的 API 密码：程序每次调用 API，都要带上它，OpenAI 才知道是谁在调用、该从谁的账户扣钱。

在开发者平台的 **API keys** 页面，点击 **Create new secret key**，起个名字（比如 `agent-book`），然后把生成的 Key 复制下来。

> ⚠️ Key 只会完整显示这一次，关掉窗口就看不到了。如果忘了，只能删掉重新创建一个。

**一定要保管好你的 Key：**

- 不要直接写在代码里，更不要上传到 GitHub。网上有程序专门扫描泄露的 Key，被别人拿去用，花的是你的钱。
- 如果怀疑 Key 泄露了，马上在 API keys 页面把它删掉。

那么不写在代码里，程序怎么拿到 Key 呢？答案是：**环境变量**。

## 准备工作 3：把 Key 保存到环境变量

环境变量是操作系统帮我们保存的一些“名字 = 值”。程序运行时可以按名字读取它们。我们把 Key 存在一个叫 `OPENAI_API_KEY` 的环境变量里，代码里只写这个名字，Key 本身就不会出现在代码中了。

**macOS / Linux**

打开终端，运行下面的命令（把 `sk-...` 换成你自己的 Key）：

```bash
echo 'export OPENAI_API_KEY="sk-..."' >> ~/.zshrc
source ~/.zshrc
```

macOS 默认用的是 zsh，所以写进 `~/.zshrc`；如果你用的是 bash，就把两处 `~/.zshrc` 换成 `~/.bashrc`。

**Windows**

打开 PowerShell，运行：

```powershell
setx OPENAI_API_KEY "sk-..."
```

`setx` 只对**新打开的**窗口生效，所以运行完要关掉 PowerShell 再重新打开。

**检查一下**

设置好以后，**重新启动 Jupyter**（Jupyter 启动时才会读取环境变量），然后运行下面这行代码。看到 `True` 就说明设置成功了。注意我们只检查 Key 在不在，不把它打印出来。

In [ ]:
import os

print("OPENAI_API_KEY" in os.environ)

最后，安装 OpenAI 官方的 Python 库：

```bash
pip install openai
```

准备工作完成，开始写代码。

## 第一次调用 API

先导入需要用到的模块。`OpenAI` 是客户端类，所有的 API 调用都通过它来发；`ChatCompletionUserMessageParam` 用来表示“用户说的一句话”，后面会解释。

In [ ]:
from openai import OpenAI
from openai.types.chat import ChatCompletionUserMessageParam

接着创建一个客户端：

- `api_key`：从环境变量里读出 Key。
- `base_url`：API 服务器的地址。写 `None` 表示用 OpenAI 官方的地址。很多其他厂商的大模型也提供了和 OpenAI 一样格式的 API，只要把 `base_url` 换成它们的地址（Key 也换成它们的），同样的代码就能调用别家的模型。

In [ ]:
client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=None
)

现在，替小明向大模型发出第一句话。调用 `client.chat.completions.create`，告诉它两件事：

- `model`：用哪个模型，这里用 `gpt-5.4-mini`。
- `messages`：对话内容，是一个列表。目前只有一条消息：`role="user"` 表示这句话是用户说的，`content` 是说话的内容。

`ChatCompletionUserMessageParam(role="user", content="...")` 其实就是一个字典 `{"role": "user", "content": "..."}`，用这个类型来写，编辑器能帮我们检查字段有没有写错。

In [ ]:
response = client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[ChatCompletionUserMessageParam(role="user", content="你好，我叫小明，正在自学编程。请用两三句话介绍一下你自己。")]
)

API 返回的 `response` 里装了不少信息。模型的回答在 `response.choices[0].message.content`：

- `choices`：一个列表。API 可以一次生成好几个候选回答，默认只有一个，所以取第 0 个。
- `message.content`：这个回答的文字内容。

In [ ]:
print(response.choices[0].message.content)

`response` 里还有一项很实用的信息：`usage`，记录了这次调用用了多少 token。还记得吗？API 就是按 token 收费的。

- `prompt_tokens`：我们发过去的内容（输入）有多少 token。
- `completion_tokens`：模型生成的回答（输出）有多少 token。

In [ ]:
print(response.usage.prompt_tokens, response.usage.completion_tokens)

## 动手试试

- 把 `content` 换成你自己的问题，比如“帮我解释一下什么是 API”。
- 连续运行两次，问“我叫什么名字？”。模型答不上来：每次调用都是独立的，它根本不知道上一次你说过什么。这正是小明说的“它记不住我”，我们会在第 5 章解决这个问题。

## 小结

- 聊天机器人只会“说”，智能体在大模型外面加上记忆、工具和规划，能帮我们“做”事情。
- 程序通过 **API** 和大模型对话。使用 API 需要开发者账号、充值，以及一个 **API Key**。
- Key 要保存在**环境变量**里，不要写进代码。
- 一次调用的核心就三样：`model`（用哪个模型）、`messages`（说了什么）、`response.choices[0].message.content`（模型的回答）。

小明的学习助手现在能说出第一句话了。不过除了 OpenAI，Anthropic 的 Claude 也是常用的大模型，两家的 API 长得不太一样。下一章，我们给不同厂商的 API 包一层统一的外壳，让程序可以随时换用不同的模型。